# Cart Abandonment Pattern Analysis
## Notebook 2: Data Cleaning & Feature Engineering Pipeline

**Role:** Senior Data Analyst  
**Objective:** Transform raw transactional logs (`kz.csv`) into a high-integrity, enriched dataset (`kz_clean.csv`) prepared for SQL, Excel, Power BI, and statistical modeling.

### Pipeline Outline:
1. Load Raw Data
2. Deduplication
3. Datetime Parsing & Epoch Filter
4. Robust Null Imputation
5. Price Outlier Treatment (1st - 99th percentile capping)
6. Feature Engineering:
   - Temporal attributes: `hour`, `day_of_week`, `time_bucket`
   - Order-level aggregation: `cart_value`
   - Funnel stage modeling: `checkout_step`
   - Device distribution simulation: `device_type_simulated`
   - User cohort classification: `customer_type` (New vs Returning vs Anonymous)
   - Primary metric: `abandoned_flag` (1 = Abandoned, 0 = Completed)
7. Export Clean Dataset to `data/kz_clean.csv`


## 1. Import Libraries

In [14]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
print("Libraries loaded.")


Libraries loaded.


## 2. Load Raw Transactions

In [15]:
import os
raw_path = '../data/kz.csv' if os.path.exists('../data/kz.csv') else 'data/kz.csv'
print(f"Loading from {raw_path}...")
df = pd.read_csv(raw_path)
print(f"Raw record count: {len(df):,}")


Loading from ../data/kz.csv...
Raw record count: 2,633,521


## 3. Deduplication

In [16]:
initial_rows = len(df)
df = df.drop_duplicates()
removed_dups = initial_rows - len(df)
print(f"Removed {removed_dups:,} duplicate rows. Remaining: {len(df):,}")


Removed 675 duplicate rows. Remaining: 2,632,846


## 4. Datetime Normalization & Temporal Glitch Filtering

In [17]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
corrupt_mask = df['event_time'].dt.year == 1970
corrupt_count = corrupt_mask.sum()

df = df[~corrupt_mask].copy()
print(f"Filtered {corrupt_count:,} corrupt epoch-zero rows. Clean date range: {df['event_time'].min()} to {df['event_time'].max()}")


Filtered 19,631 corrupt epoch-zero rows. Clean date range: 2020-01-05 03:26:46+00:00 to 2020-11-21 10:10:30+00:00


## 5. Missing Value Treatment

In [18]:
# Check nulls before
print("Nulls before imputation:")
print(df.isnull().sum())

median_price = df['price'].median()
print(f"\nImputing missing prices with median: ${median_price:.2f}")

df['user_id'] = df['user_id'].fillna(0).astype('int64')
df['category_code'] = df['category_code'].fillna('Unknown')
df['category_id'] = df['category_id'].fillna(0).astype('int64')
df['brand'] = df['brand'].fillna('Unknown')
df['price'] = df['price'].fillna(median_price)

print("\nNulls after imputation:")
print(df.isnull().sum())


Nulls before imputation:
event_time             0
order_id               0
product_id             0
category_id       427875
category_code     607074
brand             501294
price             427875
user_id          2051027
dtype: int64

Imputing missing prices with median: $55.53

Nulls after imputation:
event_time       0
order_id         0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
dtype: int64


## 6. Price Outlier Capping (Winsorization at 1st & 99th Percentiles)

In [19]:
p01 = df['price'].quantile(0.01)
p99 = df['price'].quantile(0.99)
print(f"1st Percentile  : ${p01:.2f}")
print(f"99th Percentile : ${p99:.2f}")

df['price'] = df['price'].clip(lower=p01, upper=p99)
print(f"Winsorized price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")


1st Percentile  : $1.13
99th Percentile : $1062.48
Winsorized price range: $1.13 - $1062.48


## 7. Feature Engineering

### 7.1 Temporal Features

In [20]:
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.day_name()

def assign_time_bucket(h):
    if 5 <= h < 12:    return 'Morning'
    elif 12 <= h < 17: return 'Afternoon'
    elif 17 <= h < 21: return 'Evening'
    else:              return 'Night'

df['time_bucket'] = df['hour'].apply(assign_time_bucket)
display(df[['hour', 'day_of_week', 'time_bucket']].head())


,hour,day_of_week,time_bucket
0,11,Friday,Morning
2,14,Friday,Afternoon
4,19,Friday,Evening
5,8,Sunday,Morning
6,9,Sunday,Morning


### 7.2 Total Cart Value per Order

In [21]:
order_totals = df.groupby('order_id')['price'].sum().rename('cart_value')
df = df.merge(order_totals, on='order_id', how='left')
print(f"Average Cart Value across dataset: ${df['cart_value'].mean():.2f}")


Average Cart Value across dataset: $370.49


### 7.3 Checkout Step Derivation

In [22]:
order_depth = df.groupby('order_id')['product_id'].count().rename('interaction_count')
df = df.merge(order_depth, on='order_id', how='left')

def map_checkout_stage(count):
    if count == 1:   return 'Cart'
    elif count == 2: return 'Login/Address'
    elif count == 3: return 'Shipping'
    elif count == 4: return 'Payment'
    else:            return 'Order Confirmation'

df['checkout_step'] = df['interaction_count'].apply(map_checkout_stage)
print("Checkout Step Distribution:")
print(df['checkout_step'].value_counts())


Checkout Step Distribution:
checkout_step
Cart                  867291
Login/Address         571762
Order Confirmation    532927
Shipping              390087
Payment               251148
Name: count, dtype: int64


### 7.4 Simulated Device Segmentation

In [ ]:
# Benchmark distribution: 55% Mobile, 35% Desktop, 10% Tablet
devices = ['Mobile', 'Desktop', 'Tablet']
probs = [0.55, 0.35, 0.10]
df['device_type_simulated'] = np.random.choice(devices, size=len(df), p=probs)
print("Device segmentation summary:")
print(df['device_type_simulated'].value_counts(normalize=True).round(3) * 100)


### 7.5 Customer Cohort Classification (New vs Returning)

In [ ]:
first_orders = df[df['user_id'] != 0].groupby('user_id')['order_id'].min().rename('first_order_id')
df = df.merge(first_orders, on='user_id', how='left')

def classify_customer(row):
    if row['user_id'] == 0:
        return 'Anonymous'
    return 'New' if row['order_id'] == row['first_order_id'] else 'Returning'

df['customer_type'] = df.apply(classify_customer, axis=1)
print("Customer Cohort breakdown:")
print(df['customer_type'].value_counts())


### 7.6 Primary Target Variable: Abandonment Flag

In [ ]:
df['abandoned_flag'] = (df['checkout_step'] != 'Order Confirmation').astype(int)

overall_ab_rate = df['abandoned_flag'].mean() * 100
print(f"Overall Row-level Abandonment Rate: {overall_ab_rate:.2f}%")
print(f"Abandoned Rows: {df['abandoned_flag'].sum():,} | Completed Rows: {(df['abandoned_flag'] == 0).sum():,}")


## 8. Export Clean Dataset

In [ ]:
out_path = '../data/kz_clean.csv' if os.path.exists('../data') else 'data/kz_clean.csv'
print(f"Exporting to {out_path}...")
df.to_csv(out_path, index=False)
print("Data export complete!")
